# Grammar Confidence / Internal Representation Experiment — Revised Notebook

この改訂版は、元ノートブックの以下の弱点をまとめて修正した実験ノートです。

- システムプロンプトを既定では **使わない** 形に変更
- **同系列・同サイズ** の比較を可能にするモデル設定
- データ数を増やし、**誤りタイプを拡張**
- **複数正解** を許す評価へ変更
- hidden state を **単一トークンではなく複数位置・複数 pooling** で取得
- アンカー法を **別手法（対比方向・簡易probe）でも検証**
- 正文への過剰修正を **質的に分類**
- ライブラリとモデル設定を **明示的に固定**

このノートブックは、**A: 誤りがあると断定する指示** と **B: 誤りがあれば修正する条件付き指示** の差が、
1. 出力、
2. 内部表現、
3. 過剰修正の質
にどう現れるかを、元版より妥当な形で比較するためのものです。

## 1. 依存ライブラリを固定する

再現性を上げるため、主要ライブラリはバージョンを固定します。  
ノートブックを最初に実行する環境では、このセルを先に動かしてください。

In [ ]:
# 依存ライブラリを固定
!pip -q install \
  "transformers==4.46.3" \
  "accelerate==1.1.1" \
  "sentence-transformers==3.2.1" \
  "scikit-learn==1.5.2" \
  "pandas==2.2.3" \
  "matplotlib==3.9.2" \
  "scipy==1.14.1" \
  "numpy==1.26.4" \
  "sacrebleu==2.4.3"

## 2. 基本ライブラリを読み込み、乱数と表示設定を固定する

このセルでは、実験全体で使うライブラリを読み込み、乱数・表示・保存先をそろえます。  
生成は greedy にして、環境差を最小化します。

In [ ]:
import os
import re
import gc
import json
import math
import random
import hashlib
import warnings
from dataclasses import dataclass
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import wilcoxon

from sacrebleu.metrics import CHRF
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 200)

SAVE_DIR = "grammar_confidence_revised_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 3. 実験設定を定義する

ここでは、
- モデル一覧
- 指示条件
- システムプロンプトの扱い
- 埋め込みモデル
- 取得する hidden state の位置
を定義します。

重要なのは、**同系列・同サイズ** で比べられるようにした点です。  
既定では system prompt を使わず、条件差をなるべく素直に観察します。

In [ ]:
MODEL_LIST = [
    {
        "name": "Qwen/Qwen2.5-0.5B",
        "label": "qwen25_0.5b_base",
        "family": "Qwen2.5",
        "size": "0.5B",
        "instruction_tuned": False,
    },
    {
        "name": "Qwen/Qwen2.5-0.5B-Instruct",
        "label": "qwen25_0.5b_instruct",
        "family": "Qwen2.5",
        "size": "0.5B",
        "instruction_tuned": True,
    },
    {
        "name": "Qwen/Qwen2.5-1.5B",
        "label": "qwen25_1.5b_base",
        "family": "Qwen2.5",
        "size": "1.5B",
        "instruction_tuned": False,
    },
    {
        "name": "Qwen/Qwen2.5-1.5B-Instruct",
        "label": "qwen25_1.5b_instruct",
        "family": "Qwen2.5",
        "size": "1.5B",
        "instruction_tuned": True,
    },
]

PROMPT_CONDITIONS = [
    {
        "condition": "A_assert_error",
        "instruction_ja": "以下の英文には文法的な誤りがあります。英文だけを返してください。",
        "instruction_en": "The following English sentence contains a grammatical error. Return only the corrected English sentence."
    },
    {
        "condition": "B_if_error",
        "instruction_ja": "以下の英文に文法的な誤りがあれば修正し、なければそのまま返してください。英文だけを返してください。",
        "instruction_en": "If the following English sentence contains a grammatical error, correct it; otherwise return it unchanged. Return only the English sentence."
    },
]

# システムプロンプトは既定で無効
USE_SYSTEM_PROMPT = False
NEUTRAL_SYSTEM_PROMPT = (
    "You are an English grammar correction assistant. "
    "Return only the final English sentence with no explanation."
)

GENERATION_CONFIG = {
    "max_new_tokens": 96,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
    "use_cache": True,
    "return_dict_in_generate": True,
    "output_scores": True,
    "output_hidden_states": False,
}

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHRF_METRIC = CHRF(word_order=2)

# hidden state 取得の比較対象
REPRESENTATION_SPECS = [
    "prompt_last_token",
    "prompt_sentence_span_mean",
    "prompt_instruction_span_mean",
    "generated_first_token",
    "generated_last_token",
    "generated_all_mean",
]

# 層は全層保存もできるが、分析では後段中心を見る
LAST_N_LAYERS_FOR_SUMMARY = 8
print("models:", [m["label"] for m in MODEL_LIST])

## 4. データセットを拡張する

元版より例文数を増やし、誤りタイプも広げます。  
各例には以下を持たせます。

- `sentence`: 入力文
- `references`: 許容する複数正解
- `has_error`: 元文に文法誤りがあるか
- `error_type`: 主な誤りタイプ
- `difficulty`: ざっくりした難易度

正文にも `references` を持たせ、軽微な表記揺れを許容できるようにします。

In [ ]:
DATA = [
    # subject-verb agreement
    {"id":"e01","sentence":"She go to school every day.","references":["She goes to school every day."],"has_error":True,"error_type":"subject_verb_agreement","difficulty":"easy"},
    {"id":"e02","sentence":"My brother play tennis on Sundays.","references":["My brother plays tennis on Sundays."],"has_error":True,"error_type":"subject_verb_agreement","difficulty":"easy"},
    {"id":"e03","sentence":"The students in this class studies very hard.","references":["The students in this class study very hard."],"has_error":True,"error_type":"subject_verb_agreement","difficulty":"medium"},
    {"id":"c01","sentence":"She goes to school every day.","references":["She goes to school every day."],"has_error":False,"error_type":"correct","difficulty":"easy"},
    {"id":"c02","sentence":"My brother plays tennis on Sundays.","references":["My brother plays tennis on Sundays."],"has_error":False,"error_type":"correct","difficulty":"easy"},

    # tense / aspect
    {"id":"e04","sentence":"I have ate breakfast already.","references":["I have already eaten breakfast.","I have eaten breakfast already."],"has_error":True,"error_type":"tense_aspect","difficulty":"easy"},
    {"id":"e05","sentence":"They was watching TV when I arrived.","references":["They were watching TV when I arrived."],"has_error":True,"error_type":"tense_aspect","difficulty":"easy"},
    {"id":"e06","sentence":"He didn't went to work yesterday.","references":["He didn't go to work yesterday."],"has_error":True,"error_type":"tense_aspect","difficulty":"easy"},
    {"id":"e07","sentence":"By the time we got there, the movie already started.","references":["By the time we got there, the movie had already started."],"has_error":True,"error_type":"tense_aspect","difficulty":"medium"},
    {"id":"c03","sentence":"They were watching TV when I arrived.","references":["They were watching TV when I arrived."],"has_error":False,"error_type":"correct","difficulty":"easy"},
    {"id":"c04","sentence":"By the time we got there, the movie had already started.","references":["By the time we got there, the movie had already started."],"has_error":False,"error_type":"correct","difficulty":"medium"},

    # article / determiner
    {"id":"e08","sentence":"She bought new umbrella yesterday.","references":["She bought a new umbrella yesterday."],"has_error":True,"error_type":"article","difficulty":"easy"},
    {"id":"e09","sentence":"I am looking for information about history of Japan.","references":["I am looking for information about the history of Japan."],"has_error":True,"error_type":"article","difficulty":"medium"},
    {"id":"e10","sentence":"He is teacher at local high school.","references":["He is a teacher at a local high school.","He is a teacher at the local high school."],"has_error":True,"error_type":"article","difficulty":"medium"},
    {"id":"c05","sentence":"She bought a new umbrella yesterday.","references":["She bought a new umbrella yesterday."],"has_error":False,"error_type":"correct","difficulty":"easy"},
    {"id":"c06","sentence":"He is a teacher at a local high school.","references":["He is a teacher at a local high school."],"has_error":False,"error_type":"correct","difficulty":"medium"},

    # preposition
    {"id":"e11","sentence":"We discussed about the plan yesterday.","references":["We discussed the plan yesterday."],"has_error":True,"error_type":"preposition","difficulty":"easy"},
    {"id":"e12","sentence":"She is good in math.","references":["She is good at math."],"has_error":True,"error_type":"preposition","difficulty":"easy"},
    {"id":"e13","sentence":"I am interested on learning more about this topic.","references":["I am interested in learning more about this topic."],"has_error":True,"error_type":"preposition","difficulty":"easy"},
    {"id":"c07","sentence":"We discussed the plan yesterday.","references":["We discussed the plan yesterday."],"has_error":False,"error_type":"correct","difficulty":"easy"},
    {"id":"c08","sentence":"I am interested in learning more about this topic.","references":["I am interested in learning more about this topic."],"has_error":False,"error_type":"correct","difficulty":"easy"},

    # plural / countability
    {"id":"e14","sentence":"There are many informations on the website.","references":["There is a lot of information on the website.","There is much information on the website."],"has_error":True,"error_type":"countability","difficulty":"medium"},
    {"id":"e15","sentence":"She gave me two advices.","references":["She gave me two pieces of advice.","She gave me two bits of advice."],"has_error":True,"error_type":"countability","difficulty":"medium"},
    {"id":"e16","sentence":"These equipment are expensive.","references":["This equipment is expensive."],"has_error":True,"error_type":"countability","difficulty":"medium"},
    {"id":"c09","sentence":"There is a lot of information on the website.","references":["There is a lot of information on the website."],"has_error":False,"error_type":"correct","difficulty":"medium"},
    {"id":"c10","sentence":"This equipment is expensive.","references":["This equipment is expensive."],"has_error":False,"error_type":"correct","difficulty":"medium"},

    # word order / syntax
    {"id":"e17","sentence":"What means this word?","references":["What does this word mean?"],"has_error":True,"error_type":"word_order","difficulty":"easy"},
    {"id":"e18","sentence":"I don't know where is he.","references":["I don't know where he is."],"has_error":True,"error_type":"word_order","difficulty":"easy"},
    {"id":"e19","sentence":"Rarely I have seen such a view.","references":["Rarely have I seen such a view.","I have rarely seen such a view."],"has_error":True,"error_type":"word_order","difficulty":"hard"},
    {"id":"c11","sentence":"I don't know where he is.","references":["I don't know where he is."],"has_error":False,"error_type":"correct","difficulty":"easy"},
    {"id":"c12","sentence":"I have rarely seen such a view.","references":["I have rarely seen such a view.","Rarely have I seen such a view."],"has_error":False,"error_type":"correct","difficulty":"hard"},

    # gerund / infinitive / form
    {"id":"e20","sentence":"She suggested to go earlier.","references":["She suggested going earlier."],"has_error":True,"error_type":"verb_form","difficulty":"easy"},
    {"id":"e21","sentence":"I made him to apologize.","references":["I made him apologize."],"has_error":True,"error_type":"verb_form","difficulty":"easy"},
    {"id":"e22","sentence":"This book is worth to read.","references":["This book is worth reading."],"has_error":True,"error_type":"verb_form","difficulty":"medium"},
    {"id":"c13","sentence":"She suggested going earlier.","references":["She suggested going earlier."],"has_error":False,"error_type":"correct","difficulty":"easy"},
    {"id":"c14","sentence":"This book is worth reading.","references":["This book is worth reading."],"has_error":False,"error_type":"correct","difficulty":"medium"},

    # pronoun / agreement
    {"id":"e23","sentence":"Each student must bring their pencil and submit his homework on time.","references":["Each student must bring their pencil and submit their homework on time.","Each student must bring a pencil and submit their homework on time."],"has_error":True,"error_type":"pronoun_agreement","difficulty":"hard"},
    {"id":"e24","sentence":"Neither of the answers are correct.","references":["Neither of the answers is correct."],"has_error":True,"error_type":"pronoun_agreement","difficulty":"medium"},
    {"id":"c15","sentence":"Neither of the answers is correct.","references":["Neither of the answers is correct."],"has_error":False,"error_type":"correct","difficulty":"medium"},

    # comparative / modifier
    {"id":"e25","sentence":"This problem is more easier than that one.","references":["This problem is easier than that one."],"has_error":True,"error_type":"comparative","difficulty":"easy"},
    {"id":"e26","sentence":"He explained the rule enough clearly for everyone to understand.","references":["He explained the rule clearly enough for everyone to understand."],"has_error":True,"error_type":"modifier","difficulty":"medium"},
    {"id":"c16","sentence":"This problem is easier than that one.","references":["This problem is easier than that one."],"has_error":False,"error_type":"correct","difficulty":"easy"},

    # punctuation / capitalization / sentence boundary
    {"id":"e27","sentence":"Although it was late, but we kept working.","references":["Although it was late, we kept working."],"has_error":True,"error_type":"clause_linking","difficulty":"medium"},
    {"id":"e28","sentence":"Because the train was delayed we missed the meeting.","references":["Because the train was delayed, we missed the meeting."],"has_error":True,"error_type":"punctuation","difficulty":"easy"},
    {"id":"c17","sentence":"Because the train was delayed, we missed the meeting.","references":["Because the train was delayed, we missed the meeting."],"has_error":False,"error_type":"correct","difficulty":"easy"},

    # lexicalized grammar / collocation boundary
    {"id":"e29","sentence":"She married with a doctor last year.","references":["She married a doctor last year."],"has_error":True,"error_type":"collocation_grammar","difficulty":"medium"},
    {"id":"e30","sentence":"I am used to wake up early.","references":["I am used to waking up early."],"has_error":True,"error_type":"collocation_grammar","difficulty":"medium"},
    {"id":"c18","sentence":"She married a doctor last year.","references":["She married a doctor last year."],"has_error":False,"error_type":"correct","difficulty":"medium"},

    # complex but grammatical controls
    {"id":"c19","sentence":"Had I known about the delay, I would have left earlier.","references":["Had I known about the delay, I would have left earlier."],"has_error":False,"error_type":"correct","difficulty":"hard"},
    {"id":"c20","sentence":"No sooner had the meeting started than the fire alarm went off.","references":["No sooner had the meeting started than the fire alarm went off."],"has_error":False,"error_type":"correct","difficulty":"hard"},
    {"id":"c21","sentence":"The report, which was revised twice, was finally approved.","references":["The report, which was revised twice, was finally approved."],"has_error":False,"error_type":"correct","difficulty":"medium"},
    {"id":"c22","sentence":"Whether the policy is effective remains unclear.","references":["Whether the policy is effective remains unclear."],"has_error":False,"error_type":"correct","difficulty":"medium"},
    {"id":"c23","sentence":"Only after the audit ended did they discover the discrepancy.","references":["Only after the audit ended did they discover the discrepancy."],"has_error":False,"error_type":"correct","difficulty":"hard"},
    {"id":"c24","sentence":"It is essential that every applicant be informed of the change.","references":["It is essential that every applicant be informed of the change."],"has_error":False,"error_type":"correct","difficulty":"hard"},
]

df_data = pd.DataFrame(DATA)
print("n_examples:", len(df_data))
display(df_data.head(12))
print(df_data["has_error"].value_counts())
print(df_data["error_type"].value_counts().head(15))

## 5. 補助関数を定義する

このセルでは、前処理・類似度・トークン操作・簡易差分計算など、評価と表現抽出に必要な基本関数を定義します。

In [ ]:
def safe_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x)

def simple_normalize_text(text: str) -> str:
    text = safe_text(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_for_compare(text: str) -> str:
    text = simple_normalize_text(text)
    text = text.replace("’", "'").replace("`", "'")
    text = text.replace("“", '"').replace("”", '"')
    return text

def normalize_vec(v, eps=1e-8):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v if n < eps else (v / n)

def cosine_sim(a, b) -> float:
    a = np.asarray(a).reshape(1, -1)
    b = np.asarray(b).reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])

def token_list(text: str) -> List[str]:
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\w\s]", safe_text(text))

def lexical_change_ratio(src: str, pred: str) -> float:
    s = token_list(src.lower())
    p = token_list(pred.lower())
    if not s and not p:
        return 0.0
    cs = Counter(s)
    cp = Counter(p)
    overlap = sum((cs & cp).values())
    total = max(len(s), len(p), 1)
    return 1.0 - overlap / total

def punctuation_only_change(src: str, pred: str) -> bool:
    a = re.sub(r"[^\w\s]", "", src).lower().split()
    b = re.sub(r"[^\w\s]", "", pred).lower().split()
    return a == b and normalize_for_compare(src) != normalize_for_compare(pred)

def style_only_change(src: str, pred: str) -> bool:
    a = re.sub(r"[^\w\s]", " ", src).lower().split()
    b = re.sub(r"[^\w\s]", " ", pred).lower().split()
    if not a or not b:
        return False
    return set(a) == set(b) and src != pred

def max_semantic_similarity(pred: str, refs: List[str], embed_model) -> float:
    pred = safe_text(pred)
    pred_vec = normalize_vec(embed_model.encode(pred, convert_to_numpy=True))
    sims = []
    for ref in refs:
        ref_vec = normalize_vec(embed_model.encode(ref, convert_to_numpy=True))
        sims.append(cosine_sim(pred_vec, ref_vec))
    return max(sims) if sims else 0.0

def max_chrf(pred: str, refs: List[str]) -> float:
    scores = [CHRF_METRIC.sentence_score(pred, [ref]).score for ref in refs]
    return max(scores) if scores else 0.0

def exact_match_any(pred: str, refs: List[str]) -> bool:
    p = normalize_for_compare(pred)
    return any(p == normalize_for_compare(r) for r in refs)

def hash_list(x):
    return hashlib.sha256(json.dumps(x, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:12]

## 6. モデル読み込み関数を定義する

モデルごとに tokenizer / model / config を束ねて扱います。  
`trust_remote_code=True` を使うため、実運用では利用環境のポリシーに合わせて調整してください。

In [ ]:
@dataclass
class LoadedModel:
    config: Dict[str, Any]
    tokenizer: Any
    model: Any
    hf_config: Any

def load_model_bundle(model_config: Dict[str, Any]) -> LoadedModel:
    hf_config = AutoConfig.from_pretrained(model_config["name"], trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(model_config["name"], trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_config["name"],
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    return LoadedModel(
        config=model_config,
        tokenizer=tokenizer,
        model=model,
        hf_config=hf_config,
    )

def cleanup_bundle(bundle: LoadedModel):
    try:
        del bundle.model
        del bundle.tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 7. プロンプト整形と生成関数を定義する

ここでは、モデル種別に応じて
- base モデル: プレーンテキスト
- instruct モデル: chat template
を使い分けます。

システムプロンプトは既定で **使いません**。  
使う場合も、誤りの有無を示唆しない **中立文** に限定します。

In [ ]:
def build_user_prompt(instruction: str, sentence: str) -> str:
    return f"{instruction}\n\nSentence: {sentence}\nAnswer:"

def format_prompt(bundle: LoadedModel, user_prompt: str, use_system_prompt: bool = USE_SYSTEM_PROMPT) -> str:
    tokenizer = bundle.tokenizer
    if bundle.config["instruction_tuned"] and hasattr(tokenizer, "apply_chat_template"):
        messages = []
        if use_system_prompt:
            messages.append({"role": "system", "content": NEUTRAL_SYSTEM_PROMPT})
        messages.append({"role": "user", "content": user_prompt})
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        if use_system_prompt:
            return f"System: {NEUTRAL_SYSTEM_PROMPT}\nUser: {user_prompt}\nAssistant:"
        return user_prompt

def locate_subsequence(full_ids: List[int], sub_ids: List[int]) -> Tuple[int, int]:
    if not sub_ids or len(sub_ids) > len(full_ids):
        return (-1, -1)
    for i in range(len(full_ids) - len(sub_ids) + 1):
        if full_ids[i:i+len(sub_ids)] == sub_ids:
            return (i, i + len(sub_ids))
    return (-1, -1)

def generate_text(bundle: LoadedModel, prompt_text: str):
    tokenizer = bundle.tokenizer
    model = bundle.model

    inputs = tokenizer(prompt_text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        gen = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **GENERATION_CONFIG
        )

    full_ids = gen.sequences[0]
    generated_ids = full_ids[input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    token_logprobs = []
    if len(gen.scores) > 0:
        for step, scores in enumerate(gen.scores):
            token_id = generated_ids[step]
            logp = torch.log_softmax(scores[0], dim=-1)[token_id].item()
            token_logprobs.append(logp)

    mean_logprob = float(np.mean(token_logprobs)) if token_logprobs else np.nan

    return {
        "prompt_text": prompt_text,
        "full_ids": full_ids.detach().cpu(),
        "input_ids": input_ids[0].detach().cpu(),
        "generated_ids": generated_ids.detach().cpu(),
        "generated_text": generated_text,
        "token_logprobs": token_logprobs,
        "mean_logprob": mean_logprob,
    }

## 8. hidden state を複数位置から抽出する関数を定義する

元版では最後の 1 トークンに大きく依存していました。  
ここでは以下の複数表現を取ります。

- `prompt_last_token`
- `prompt_sentence_span_mean`
- `prompt_instruction_span_mean`
- `generated_first_token`
- `generated_last_token`
- `generated_all_mean`

これにより、句点や EOS 近傍に過度に引っ張られる問題を減らします。

In [ ]:
def _mean_pool_hidden(hidden: torch.Tensor, start: int, end: int):
    if start < 0 or end <= start or end > hidden.shape[0]:
        return None
    return hidden[start:end].mean(dim=0).detach().cpu().numpy()

def extract_representations(
    bundle: LoadedModel,
    prompt_text: str,
    sentence_text: str,
    instruction_text: str,
    generation_result: Dict[str, Any]
) -> Dict[int, Dict[str, np.ndarray]]:
    tokenizer = bundle.tokenizer
    model = bundle.model

    full_ids = generation_result["full_ids"].unsqueeze(0).to(model.device)
    input_ids = generation_result["input_ids"].tolist()
    full_ids_list = generation_result["full_ids"].tolist()

    sentence_ids = tokenizer(sentence_text, add_special_tokens=False)["input_ids"]
    instruction_ids = tokenizer(instruction_text, add_special_tokens=False)["input_ids"]

    sent_start, sent_end = locate_subsequence(input_ids, sentence_ids)
    inst_start, inst_end = locate_subsequence(input_ids, instruction_ids)

    with torch.no_grad():
        outputs = model(full_ids, output_hidden_states=True, use_cache=False)

    hidden_states = outputs.hidden_states  # embedding + each layer
    reps_by_layer = {}

    prompt_len = len(input_ids)
    gen_len = len(full_ids_list) - prompt_len

    for layer_idx, hs in enumerate(hidden_states):
        hs = hs[0]  # [seq, hidden]
        reps = {}

        if prompt_len > 0:
            reps["prompt_last_token"] = hs[prompt_len - 1].detach().cpu().numpy()

        x = _mean_pool_hidden(hs, sent_start, sent_end)
        if x is not None:
            reps["prompt_sentence_span_mean"] = x

        x = _mean_pool_hidden(hs, inst_start, inst_end)
        if x is not None:
            reps["prompt_instruction_span_mean"] = x

        if gen_len > 0:
            reps["generated_first_token"] = hs[prompt_len].detach().cpu().numpy()
            reps["generated_last_token"] = hs[prompt_len + gen_len - 1].detach().cpu().numpy()
            reps["generated_all_mean"] = hs[prompt_len:prompt_len + gen_len].mean(dim=0).detach().cpu().numpy()

        reps_by_layer[layer_idx] = reps

    return reps_by_layer

## 9. アンカー法を改善する

アンカーは 1 つの方法だけでは不安定なので、3 本立てで検証します。

1. **手作りアンカー平均**
2. **対比方向ベクトル**（assertive vs conditional などの平均差）
3. **簡易 probe**（小規模キャリブレーション文を使ったロジスティック回帰）

このセルでは、まずキャリブレーション用の文セットを定義します。

In [ ]:
ANCHOR_TEXTS = {
    "assertive_correction": [
        "The sentence definitely contains a grammatical error and must be corrected.",
        "A grammatical mistake is present, so the sentence should be fixed.",
        "This input requires correction."
    ],
    "conditional_correction": [
        "If the sentence contains an error, correct it; otherwise leave it unchanged.",
        "The sentence may already be correct, so correction is conditional.",
        "Only correct the sentence when an error is actually present."
    ],
    "high_confidence": [
        "I am certain that this sentence should be corrected.",
        "The need for correction is clear and unambiguous.",
        "The model can confidently decide to edit."
    ],
    "low_confidence": [
        "The sentence may or may not need correction.",
        "It is uncertain whether an edit is necessary.",
        "The model should remain cautious about making a change."
    ],
}

CALIBRATION_TEXTS = [
    {"label_set":"assertive_vs_conditional", "label":1, "text":"The sentence definitely contains an error, so correct it."},
    {"label_set":"assertive_vs_conditional", "label":1, "text":"A grammatical mistake is certainly present and must be fixed."},
    {"label_set":"assertive_vs_conditional", "label":1, "text":"Correct this sentence because it is definitely wrong."},
    {"label_set":"assertive_vs_conditional", "label":0, "text":"If the sentence is wrong, correct it; otherwise leave it unchanged."},
    {"label_set":"assertive_vs_conditional", "label":0, "text":"Only edit the sentence when an error is actually present."},
    {"label_set":"assertive_vs_conditional", "label":0, "text":"The sentence may already be correct, so correction is conditional."},

    {"label_set":"confidence", "label":1, "text":"I am certain that an edit is needed."},
    {"label_set":"confidence", "label":1, "text":"The correction decision is clear."},
    {"label_set":"confidence", "label":1, "text":"The need to revise is obvious."},
    {"label_set":"confidence", "label":0, "text":"It is unclear whether revision is needed."},
    {"label_set":"confidence", "label":0, "text":"The model should stay cautious before editing."},
    {"label_set":"confidence", "label":0, "text":"It may be unnecessary to make any change."},
]
pd.DataFrame(CALIBRATION_TEXTS)

## 10. アンカー・方向ベクトル・probe を構築する関数を定義する

このセルでは、各モデルごとに
- テキストから表現を抽出し、
- アンカー平均ベクトルを作り、
- 対比方向ベクトルを作り、
- probe を学習する
関数を定義します。

probe は **補助的** な検証用であり、強い因果主張をするためのものではありません。

In [ ]:
def get_text_representation(bundle: LoadedModel, text: str, layer_pick: str = "last", rep_name: str = "prompt_last_token") -> np.ndarray:
    prompt_text = format_prompt(bundle, text, use_system_prompt=USE_SYSTEM_PROMPT)
    gen = generate_text(bundle, prompt_text)
    reps = extract_representations(bundle, prompt_text, sentence_text=text, instruction_text=text, generation_result=gen)

    layer_idx = max(reps.keys()) if layer_pick == "last" else int(layer_pick)
    vec = reps[layer_idx].get(rep_name)
    if vec is None:
        # fallback
        for alt in ["prompt_sentence_span_mean", "prompt_instruction_span_mean", "prompt_last_token"]:
            if alt in reps[layer_idx]:
                vec = reps[layer_idx][alt]
                break
    return normalize_vec(vec)

def build_anchor_package(bundle: LoadedModel):
    anchor_vectors = {}
    for name, texts in ANCHOR_TEXTS.items():
        vecs = [get_text_representation(bundle, t, rep_name="prompt_last_token") for t in texts]
        anchor_vectors[name] = normalize_vec(np.mean(vecs, axis=0))

    direction_vectors = {
        "assertive_minus_conditional": normalize_vec(
            anchor_vectors["assertive_correction"] - anchor_vectors["conditional_correction"]
        ),
        "high_minus_low_confidence": normalize_vec(
            anchor_vectors["high_confidence"] - anchor_vectors["low_confidence"]
        )
    }

    probe_models = {}
    df_cal = pd.DataFrame(CALIBRATION_TEXTS)
    for label_set in df_cal["label_set"].unique():
        sub = df_cal[df_cal["label_set"] == label_set].copy()
        X = np.vstack([get_text_representation(bundle, t, rep_name="prompt_last_token") for t in sub["text"]])
        y = sub["label"].values
        probe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(random_state=SEED, max_iter=1000))
        ])
        probe.fit(X, y)
        probe_models[label_set] = probe

    return {
        "anchors": anchor_vectors,
        "directions": direction_vectors,
        "probes": probe_models,
    }

def score_representation(vec: np.ndarray, anchor_package: Dict[str, Any]) -> Dict[str, float]:
    vec = normalize_vec(vec)
    scores = {
        "sim_assertive_anchor": cosine_sim(vec, anchor_package["anchors"]["assertive_correction"]),
        "sim_conditional_anchor": cosine_sim(vec, anchor_package["anchors"]["conditional_correction"]),
        "sim_high_conf_anchor": cosine_sim(vec, anchor_package["anchors"]["high_confidence"]),
        "sim_low_conf_anchor": cosine_sim(vec, anchor_package["anchors"]["low_confidence"]),
        "dir_assertive_minus_conditional": float(np.dot(vec, anchor_package["directions"]["assertive_minus_conditional"])),
        "dir_high_minus_low_conf": float(np.dot(vec, anchor_package["directions"]["high_minus_low_confidence"])),
        "probe_assertive_prob": float(anchor_package["probes"]["assertive_vs_conditional"].predict_proba(vec.reshape(1, -1))[0, 1]),
        "probe_high_conf_prob": float(anchor_package["probes"]["confidence"].predict_proba(vec.reshape(1, -1))[0, 1]),
    }
    return scores

## 11. 出力評価と過剰修正分類を定義する

元版では exact match が強すぎました。  
ここでは複数正解を許しつつ、以下を併用します。

- `exact_match_any`
- `best_chrf`
- `best_semantic_similarity`
- `changed`
- `corrected_when_needed`
- `preserved_when_correct`

さらに、正文を変えてしまったケースを質的に分類します。

In [ ]:
HEDGING_PATTERNS = [
    r"\bmaybe\b", r"\bperhaps\b", r"\bprobably\b", r"\bit seems\b",
    r"\bi think\b", r"\bit may\b", r"\bit might\b"
]
META_PATTERNS = [
    r"\bcorrected sentence\b", r"\bhere is\b", r"\bthe corrected\b",
    r"\bgrammar\b", r"\bexplanation\b", r"\banswer\b"
]

def contains_pattern(text: str, patterns: List[str]) -> bool:
    t = safe_text(text).lower()
    return any(re.search(p, t) for p in patterns)

def classify_overcorrection(src: str, pred: str, has_error: bool) -> str:
    src_n = normalize_for_compare(src)
    pred_n = normalize_for_compare(pred)

    if src_n == pred_n:
        return "unchanged"

    if contains_pattern(pred, META_PATTERNS):
        return "meta_or_explanation"

    lcr = lexical_change_ratio(src, pred)

    if punctuation_only_change(src, pred):
        return "punctuation_only"

    if style_only_change(src, pred) and lcr < 0.35:
        return "style_or_surface_rephrasing"

    if lcr < 0.25:
        return "local_grammar_edit"

    if lcr < 0.55:
        return "lexical_or_clause_rewrite"

    return "major_rewrite"

def evaluate_output(src: str, refs: List[str], pred: str, has_error: bool, embed_model) -> Dict[str, Any]:
    pred_n = normalize_for_compare(pred)
    src_n = normalize_for_compare(src)

    changed = pred_n != src_n
    exact = exact_match_any(pred, refs)
    best_sem = max_semantic_similarity(pred, refs, embed_model)
    best_chrf = max_chrf(pred, refs)

    corrected_when_needed = bool(has_error and exact)
    preserved_when_correct = bool((not has_error) and exact)

    return {
        "changed": changed,
        "exact_match_any": exact,
        "best_semantic_similarity": best_sem,
        "best_chrf": best_chrf,
        "corrected_when_needed": corrected_when_needed,
        "preserved_when_correct": preserved_when_correct,
        "hedging_output": contains_pattern(pred, HEDGING_PATTERNS),
        "meta_output": contains_pattern(pred, META_PATTERNS),
        "lexical_change_ratio": lexical_change_ratio(src, pred),
        "edit_category": classify_overcorrection(src, pred, has_error),
    }

## 12. 実験本体を実行する

このセルで、各モデル・各条件・各例文について

1. 生成
2. 出力評価
3. hidden state 抽出
4. アンカー / 方向 / probe のスコア計算

をまとめて実行します。

実行時間は環境によって変わります。  
GPU がない場合はかなり時間がかかることがあります。

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

all_outputs = []
all_reps = []
run_manifest = []

for model_cfg in MODEL_LIST:
    print("=" * 100)
    print("Loading:", model_cfg["name"])
    bundle = load_model_bundle(model_cfg)

    anchor_package = build_anchor_package(bundle)

    manifest_row = {
        "model_label": model_cfg["label"],
        "model_name": model_cfg["name"],
        "family": model_cfg["family"],
        "size": model_cfg["size"],
        "instruction_tuned": model_cfg["instruction_tuned"],
        "use_system_prompt": USE_SYSTEM_PROMPT,
        "neutral_system_prompt": NEUTRAL_SYSTEM_PROMPT if USE_SYSTEM_PROMPT else "",
        "generation_config_hash": hash_list(GENERATION_CONFIG),
        "representation_specs_hash": hash_list(REPRESENTATION_SPECS),
        "embed_model_name": EMBED_MODEL_NAME,
        "seed": SEED,
        "torch_version": torch.__version__,
    }
    run_manifest.append(manifest_row)

    for cond in PROMPT_CONDITIONS:
        print("  condition:", cond["condition"])

        for ex in DATA:
            user_prompt = build_user_prompt(cond["instruction_en"], ex["sentence"])
            prompt_text = format_prompt(bundle, user_prompt, use_system_prompt=USE_SYSTEM_PROMPT)
            gen = generate_text(bundle, prompt_text)

            pred = simple_normalize_text(gen["generated_text"])
            eval_row = evaluate_output(ex["sentence"], ex["references"], pred, ex["has_error"], embed_model)

            out_row = {
                "model_label": model_cfg["label"],
                "model_name": model_cfg["name"],
                "family": model_cfg["family"],
                "size": model_cfg["size"],
                "instruction_tuned": model_cfg["instruction_tuned"],
                "condition": cond["condition"],
                "instruction_en": cond["instruction_en"],
                "example_id": ex["id"],
                "input_sentence": ex["sentence"],
                "references_json": json.dumps(ex["references"], ensure_ascii=False),
                "has_error": ex["has_error"],
                "error_type": ex["error_type"],
                "difficulty": ex["difficulty"],
                "generated": pred,
                "mean_logprob": gen["mean_logprob"],
                "n_generated_tokens": len(gen["generated_ids"]),
                **eval_row,
            }
            all_outputs.append(out_row)

            reps_by_layer = extract_representations(
                bundle=bundle,
                prompt_text=prompt_text,
                sentence_text=ex["sentence"],
                instruction_text=cond["instruction_en"],
                generation_result=gen
            )

            for layer_idx, rep_map in reps_by_layer.items():
                for rep_name, vec in rep_map.items():
                    score_map = score_representation(vec, anchor_package)
                    rep_row = {
                        "model_label": model_cfg["label"],
                        "model_name": model_cfg["name"],
                        "size": model_cfg["size"],
                        "instruction_tuned": model_cfg["instruction_tuned"],
                        "condition": cond["condition"],
                        "example_id": ex["id"],
                        "has_error": ex["has_error"],
                        "error_type": ex["error_type"],
                        "layer": layer_idx,
                        "representation_name": rep_name,
                        **score_map,
                    }
                    all_reps.append(rep_row)

    cleanup_bundle(bundle)

df_outputs = pd.DataFrame(all_outputs)
df_reps = pd.DataFrame(all_reps)
df_manifest = pd.DataFrame(run_manifest)

print("outputs:", df_outputs.shape)
print("reps:", df_reps.shape)
display(df_outputs.head(5))
display(df_reps.head(5))

## 13. 実験結果と設定情報を保存する

再現性のため、出力・表現・マニフェスト・データセットを保存します。

In [ ]:
outputs_csv = os.path.join(SAVE_DIR, "revised_outputs.csv")
reps_csv = os.path.join(SAVE_DIR, "revised_representations.csv")
manifest_csv = os.path.join(SAVE_DIR, "run_manifest.csv")
dataset_json = os.path.join(SAVE_DIR, "dataset.json")
settings_json = os.path.join(SAVE_DIR, "experiment_settings.json")

df_outputs.to_csv(outputs_csv, index=False, encoding="utf-8-sig")
df_reps.to_csv(reps_csv, index=False, encoding="utf-8-sig")
df_manifest.to_csv(manifest_csv, index=False, encoding="utf-8-sig")

with open(dataset_json, "w", encoding="utf-8") as f:
    json.dump(DATA, f, ensure_ascii=False, indent=2)

with open(settings_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "SEED": SEED,
            "USE_SYSTEM_PROMPT": USE_SYSTEM_PROMPT,
            "NEUTRAL_SYSTEM_PROMPT": NEUTRAL_SYSTEM_PROMPT,
            "GENERATION_CONFIG": GENERATION_CONFIG,
            "MODEL_LIST": MODEL_LIST,
            "PROMPT_CONDITIONS": PROMPT_CONDITIONS,
            "EMBED_MODEL_NAME": EMBED_MODEL_NAME,
            "REPRESENTATION_SPECS": REPRESENTATION_SPECS,
            "LAST_N_LAYERS_FOR_SUMMARY": LAST_N_LAYERS_FOR_SUMMARY,
        },
        f, ensure_ascii=False, indent=2
    )

print(outputs_csv)
print(reps_csv)
print(manifest_csv)
print(dataset_json)
print(settings_json)

## 14. 出力精度をモデル × 条件ごとに要約する

まずは全体の出力傾向を俯瞰します。  
複数正解を許した上で、誤文訂正率・正文保持率・意味類似・文字 n-gram 類似などを見ます。

In [ ]:
summary_rows = []
for (model_label, condition), sub in df_outputs.groupby(["model_label", "condition"]):
    err = sub[sub["has_error"] == True]
    cor = sub[sub["has_error"] == False]

    summary_rows.append({
        "model_label": model_label,
        "condition": condition,
        "n_examples": len(sub),
        "exact_match_any_rate": sub["exact_match_any"].mean(),
        "best_chrf_mean": sub["best_chrf"].mean(),
        "best_semantic_similarity_mean": sub["best_semantic_similarity"].mean(),
        "changed_rate": sub["changed"].mean(),
        "mean_logprob_mean": sub["mean_logprob"].mean(),
        "corrected_when_needed_rate": err["corrected_when_needed"].mean() if len(err) else np.nan,
        "preserved_when_correct_rate": cor["preserved_when_correct"].mean() if len(cor) else np.nan,
        "hedging_output_rate": sub["hedging_output"].mean(),
        "meta_output_rate": sub["meta_output"].mean(),
    })

df_summary = pd.DataFrame(summary_rows).sort_values(["model_label", "condition"]).reset_index(drop=True)
display(df_summary)

## 15. 同サイズ・同系列で base / instruct を直接比べる

ここでは size ごとに base と instruct を並べ、**同サイズ比較** を明示します。  
元版の「サイズ差」と「instruction tuning 差」が混ざる問題を避けるための要約です。

In [ ]:
base_instruct_cmp = []
for size in sorted(df_outputs["size"].unique()):
    for condition in [c["condition"] for c in PROMPT_CONDITIONS]:
        base_label = f"qwen25_{size.lower()}_base"
        inst_label = f"qwen25_{size.lower()}_instruct"
        left = df_summary[(df_summary["model_label"] == base_label) & (df_summary["condition"] == condition)]
        right = df_summary[(df_summary["model_label"] == inst_label) & (df_summary["condition"] == condition)]
        if len(left) and len(right):
            l = left.iloc[0]
            r = right.iloc[0]
            base_instruct_cmp.append({
                "size": size,
                "condition": condition,
                "base_exact": l["exact_match_any_rate"],
                "inst_exact": r["exact_match_any_rate"],
                "inst_minus_base_exact": r["exact_match_any_rate"] - l["exact_match_any_rate"],
                "base_corrected": l["corrected_when_needed_rate"],
                "inst_corrected": r["corrected_when_needed_rate"],
                "inst_minus_base_corrected": r["corrected_when_needed_rate"] - l["corrected_when_needed_rate"],
                "base_preserved": l["preserved_when_correct_rate"],
                "inst_preserved": r["preserved_when_correct_rate"],
                "inst_minus_base_preserved": r["preserved_when_correct_rate"] - l["preserved_when_correct_rate"],
            })

df_same_size_cmp = pd.DataFrame(base_instruct_cmp)
display(df_same_size_cmp)

## 16. A 条件と B 条件を対応比較する

同じ例文に対して A/B を並べ、どちらが誤文訂正や正文保持に有利かを比較します。  
検定は簡易的に Wilcoxon を使い、n が大きくないことを前提に解釈は慎重に行います。

In [ ]:
paired = df_outputs.pivot_table(
    index=["model_label", "example_id", "has_error", "error_type"],
    columns="condition",
    values=[
        "exact_match_any",
        "corrected_when_needed",
        "preserved_when_correct",
        "best_chrf",
        "best_semantic_similarity",
        "changed",
        "mean_logprob",
    ],
    aggfunc="first"
)

paired.columns = ["__".join(map(str, c)) for c in paired.columns]
paired = paired.reset_index()
display(paired.head())

wilcoxon_rows = []
for model_label, sub in paired.groupby("model_label"):
    for metric in ["exact_match_any", "best_chrf", "best_semantic_similarity", "changed", "mean_logprob"]:
        a_col = f"{metric}__A_assert_error"
        b_col = f"{metric}__B_if_error"
        vals = sub[[a_col, b_col]].dropna()
        if len(vals) >= 5:
            try:
                stat, p = wilcoxon(vals[a_col], vals[b_col], zero_method="wilcox", alternative="two-sided")
            except ValueError:
                stat, p = np.nan, np.nan
        else:
            stat, p = np.nan, np.nan

        wilcoxon_rows.append({
            "model_label": model_label,
            "metric": metric,
            "A_mean": sub[a_col].mean(),
            "B_mean": sub[b_col].mean(),
            "B_minus_A": sub[b_col].mean() - sub[a_col].mean(),
            "wilcoxon_stat": stat,
            "p_value": p,
            "n_pairs": len(vals),
        })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
display(df_wilcoxon.sort_values(["model_label", "metric"]))

## 17. 正文に対する過剰修正を質的に集計する

正文を変更してしまったケースだけを取り出し、どの種類の過剰修正が多いかを見ます。  
これにより、単なる誤判定なのか、スタイル変更なのか、説明文混入なのかを切り分けられます。

In [ ]:
over_sub = df_outputs[(df_outputs["has_error"] == False) & (df_outputs["changed"] == True)].copy()

over_summary = (
    over_sub.groupby(["model_label", "condition", "edit_category"])
    .size()
    .reset_index(name="count")
    .sort_values(["model_label", "condition", "count"], ascending=[True, True, False])
)

display(over_summary)

display(
    over_sub[
        [
            "model_label", "condition", "example_id", "input_sentence",
            "generated", "edit_category", "lexical_change_ratio",
            "best_chrf", "best_semantic_similarity"
        ]
    ].sort_values(["model_label", "condition", "edit_category"])
)

## 18. エラータイプ別の成績を見る

誤文だけを対象にして、どの誤りタイプで A/B 差が出やすいかを確認します。

In [ ]:
err_type_summary = (
    df_outputs[df_outputs["has_error"] == True]
    .groupby(["model_label", "condition", "error_type"])[
        ["corrected_when_needed", "best_chrf", "best_semantic_similarity", "mean_logprob"]
    ]
    .mean()
    .reset_index()
    .sort_values(["model_label", "error_type", "condition"])
)

display(err_type_summary.head(50))

## 19. hidden state の全体傾向を表現位置ごとに要約する

ここでは後段の層を中心に、representation ごとの平均スコアを見ます。  
単一トークンの最後だけでなく、文スパン平均や生成全体平均も比較できます。

In [ ]:
max_layer = int(df_reps["layer"].max())
layer_threshold = max(0, max_layer - LAST_N_LAYERS_FOR_SUMMARY + 1)

df_reps_tail = df_reps[df_reps["layer"] >= layer_threshold].copy()

rep_summary = (
    df_reps_tail.groupby(["model_label", "condition", "representation_name"])[
        [
            "sim_assertive_anchor",
            "sim_conditional_anchor",
            "sim_high_conf_anchor",
            "sim_low_conf_anchor",
            "dir_assertive_minus_conditional",
            "dir_high_minus_low_conf",
            "probe_assertive_prob",
            "probe_high_conf_prob",
        ]
    ]
    .mean()
    .reset_index()
    .sort_values(["model_label", "representation_name", "condition"])
)

display(rep_summary.head(100))

## 20. 層ごとの推移を可視化する

各スコアが層とともにどう動くかを見るため、主要な representation について折れ線で可視化します。

In [ ]:
plot_metrics = [
    "sim_assertive_anchor",
    "sim_conditional_anchor",
    "dir_assertive_minus_conditional",
    "probe_assertive_prob",
    "sim_high_conf_anchor",
    "sim_low_conf_anchor",
    "dir_high_minus_low_conf",
    "probe_high_conf_prob",
]

rep_targets = ["prompt_sentence_span_mean", "generated_all_mean"]

for model_label in sorted(df_reps["model_label"].unique()):
    for rep_name in rep_targets:
        sub = df_reps[df_reps["model_label"] == model_label]
        sub = sub[sub["representation_name"] == rep_name]
        if sub.empty:
            continue

        mean_df = sub.groupby(["condition", "layer"])[plot_metrics].mean().reset_index()

        for metric in plot_metrics:
            plt.figure(figsize=(8, 4))
            for condition in mean_df["condition"].unique():
                s = mean_df[mean_df["condition"] == condition]
                plt.plot(s["layer"], s[metric], label=condition)
            plt.title(f"{model_label} | {rep_name} | {metric}")
            plt.xlabel("Layer")
            plt.ylabel(metric)
            plt.legend()
            plt.show()

## 21. 表現位置ごとの安定性を比較する

同じ主張が `prompt_last_token` だけでなく、他の representation でも再現するかをざっと確認します。  
ここでは A/B の差分を representation ごとに平均化して並べます。

In [ ]:
rep_diff = (
    df_reps_tail.pivot_table(
        index=["model_label", "example_id", "layer", "representation_name"],
        columns="condition",
        values=[
            "dir_assertive_minus_conditional",
            "probe_assertive_prob",
            "dir_high_minus_low_conf",
            "probe_high_conf_prob",
        ],
        aggfunc="first"
    )
)

rep_diff.columns = ["__".join(map(str, c)) for c in rep_diff.columns]
rep_diff = rep_diff.reset_index()

rows = []
for (model_label, representation_name), sub in rep_diff.groupby(["model_label", "representation_name"]):
    row = {
        "model_label": model_label,
        "representation_name": representation_name,
    }
    for metric in [
        "dir_assertive_minus_conditional",
        "probe_assertive_prob",
        "dir_high_minus_low_conf",
        "probe_high_conf_prob",
    ]:
        a = f"{metric}__A_assert_error"
        b = f"{metric}__B_if_error"
        row[f"B_minus_A__{metric}"] = sub[b].mean() - sub[a].mean()
    rows.append(row)

df_rep_stability = pd.DataFrame(rows).sort_values(["model_label", "representation_name"])
display(df_rep_stability)

## 22. 代表例を確認する

定量指標だけでは分かりにくいので、いくつかの代表例を人手で点検しやすい形で表示します。

In [ ]:
display(
    df_outputs[
        [
            "model_label", "condition", "example_id", "has_error", "error_type",
            "input_sentence", "references_json", "generated",
            "exact_match_any", "best_chrf", "best_semantic_similarity",
            "changed", "edit_category", "mean_logprob"
        ]
    ]
    .sort_values(["model_label", "condition", "example_id"])
    .head(80)
)

## 23. 最終サマリーを自動生成する

最後に、ノートブックの主要所見をテキストでまとめます。  
これはドラフト的な要約であり、最終的には表と代表例を見て人手で整える前提です。

In [ ]:
summary_lines = []

summary_lines.append("### 実験の要点")
summary_lines.append(f"- 例文数: {len(df_data)}")
summary_lines.append(f"- 誤文数: {int(df_data['has_error'].sum())}")
summary_lines.append(f"- 正文数: {int((~df_data['has_error']).sum())}")
summary_lines.append(f"- system prompt 使用: {USE_SYSTEM_PROMPT}")
summary_lines.append("")

summary_lines.append("### 出力成績")
for _, row in df_summary.iterrows():
    summary_lines.append(
        f"- {row['model_label']} / {row['condition']}: "
        f"exact={row['exact_match_any_rate']:.3f}, "
        f"corrected={row['corrected_when_needed_rate']:.3f}, "
        f"preserved={row['preserved_when_correct_rate']:.3f}, "
        f"changed={row['changed_rate']:.3f}"
    )

summary_lines.append("")
summary_lines.append("### 正文への過剰修正")
if len(over_summary):
    for _, row in over_summary.head(20).iterrows():
        summary_lines.append(
            f"- {row['model_label']} / {row['condition']} / {row['edit_category']}: {row['count']}"
        )
else:
    summary_lines.append("- 過剰修正は観測されなかった。")

summary_lines.append("")
summary_lines.append("### 内部表現")
summary_lines.append("- `prompt_last_token` のみでなく `prompt_sentence_span_mean` と `generated_all_mean` も併記して解釈する。")
summary_lines.append("- アンカー類似度だけでなく、方向ベクトルと probe の符号・確率が整合しているか確認する。")
summary_lines.append("- ある representation だけで差が出る場合は、結論を弱めて扱う。")

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(os.path.join(SAVE_DIR, "auto_summary.md"), "w", encoding="utf-8") as f:
    f.write(summary_text)

## 24. 読み取り上の注意

この改訂版でも、なお注意すべき点があります。

- probe は補助的な検証であり、意味軸の完全な保証ではない
- 例文は拡張したが、依然として人工的な側面がある
- greedy decoding に固定しているため、サンプリング時の挙動差は見ていない
- hidden state の解釈は本質的に間接的である

それでも、元版よりは
- 条件差の混線が少なく、
- 同サイズ比較ができ、
- 複数正解を許し、
- 表現位置依存性を点検でき、
- 過剰修正の中身まで見られる
構成になっています。